# Ilera-ATLAS · 01b: can the **raw** N-ATLaS find the right Standing Orders reference?

A baseline **before any fine-tuning**. The base model `NCAIR1/N-ATLaS` is downloaded from Hugging Face,
saved to your **Google Drive** in 4-bit (~6 GB, so it fits a free 15 GB Drive and later sessions skip
the 16 GB download), and tested on questions built automatically from the raw Standing Orders chunks
(`raw/chunks.jsonl` from `01_extract`). No human review is needed: every chunk already knows its own
section number, title and pages.

| Test | The model sees | It must answer | What it tells us |
|---|---|---|---|
| `closed_book` | a topic only ("which section covers worm infestation?") | the section number | does N-ATLaS already **know** the Standing Orders? |
| `index` | an excerpt **+ the table of contents** of the document | the section number | can it **map content to its reference** when given the index (retrieval) |
| `mcq` | an excerpt + 4 section options | A/B/C/D | the easy floor: can it recognise the right section at all? (chance = 25%) |

Scores per test: `number_ok` (exact section number), `title_ok` (picked a section with the right title:
lenient, because e.g. EYE DISORDERS exists for children, adults and the elderly), and `mcq_ok`.

**Colab setup (once):**
1. Runtime → Change runtime type → **T4 GPU**.
2. Key icon (Secrets) on the left → add `HF_TOKEN` (your Hugging Face token, after accepting the
   N-ATLaS terms) → switch on *Notebook access*.
3. Run all. Approve the Google Drive pop-up.

First run: ~15–25 min (download + 4-bit save to Drive + ~470 short answers). Later runs: ~10 min.
Also runs on Kaggle (set `USE_DRIVE = False`; the model then stays on Kaggle's disk).

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"            # <- your Hugging Face username (owner of ilera-data)
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"
USE_DRIVE = True                        # save the 4-bit model + results in Google Drive (Colab)
DRIVE_DIR = "/content/drive/MyDrive/ilera"
DOCS = ["CHEW"]                         # documents to test: "CHEW", "JCHEW", "CHO"
TASKS = ["closed_book", "index", "mcq"]
LIMIT = None                            # e.g. 12 = quick smoke test (items per task)
BATCH = 8
SEED = 7
UPLOAD_RESULTS = True                   # also copy results to ilera-data/results/ on Hugging Face
LOCAL_CHUNKS = None                     # offline testing: path to a local chunks.jsonl

In [ ]:
import os, subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated default on some hosts
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # faster model download
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "rapidfuzz"],
                   capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")

In [ ]:
import os, sys
def get_hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
assert HF_TOKEN, "No HF_TOKEN: Colab key icon -> add secret HF_TOKEN -> enable Notebook access"
os.environ["HF_TOKEN"] = HF_TOKEN

HOME = "."
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        HOME = DRIVE_DIR
    except Exception as e:
        print("Google Drive not available (not on Colab?); using local disk:", repr(e)[:100])
        HOME = "/kaggle/working/ilera" if os.path.isdir("/kaggle") else "ilera"
MODEL_DIR = os.path.join(HOME, "models", "N-ATLaS-4bit")
RESULTS_DIR = os.path.join(HOME, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("model folder:", MODEL_DIR, "| results:", RESULTS_DIR)

## 1. Raw dataset → reference questions
Each **chunk** is one part of a document and carries its true reference (section number, title,
pages). For every chunk we build one `index` item and one `mcq` item from an excerpt in the middle of
the chunk, and for every **section** one `closed_book` item. Items are reproducible (`SEED`).

In [ ]:
import json, random, re, collections

def load_chunks():
    if LOCAL_CHUNKS:
        path = LOCAL_CHUNKS
    else:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(DATA_REPO, "raw/chunks.jsonl", repo_type="dataset", token=HF_TOKEN)
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

chunks = [c for c in load_chunks() if c["doc"] in DOCS]
assert chunks, f"no chunks for {DOCS}: run 01_extract first"

def sec_key(s):
    return tuple(int(x) for x in s.split("."))

sections = {}                                   # (doc, section_id) -> title
for c in chunks:
    sections.setdefault((c["doc"], c["section_id"]), " ".join(c["title"].split()))
toc = {d: "\n".join(f"{sid} {t}" for (dd, sid), t in sorted(sections.items(), key=lambda kv: sec_key(kv[0][1]))
                    if dd == d) for d in DOCS}
DOC_NAME = {"CHEW": "Community Health Extension Workers (CHEW)",
            "JCHEW": "Junior Community Health Extension Workers (JCHEW)",
            "CHO": "Community Health Officers (CHO)"}

def excerpt(text, rng, words=70):
    lines = [l for l in text.splitlines() if len(l.split()) >= 3]
    w = " ".join(lines).split()
    if len(w) <= words:
        return " ".join(w)
    start = rng.randint(len(w) // 4, max(len(w) // 4, len(w) - words - len(w) // 8))
    return " ".join(w[start:start + words])

rng = random.Random(SEED)
items = []
if "closed_book" in TASKS:
    # one question per distinct title: e.g. EYE DISORDERS exists for children, adults and the elderly,
    # so any of its section numbers counts as correct
    by_title = collections.defaultdict(list)
    for (doc, sid), title in sorted(sections.items(), key=lambda kv: (kv[0][0], sec_key(kv[0][1]))):
        by_title[(doc, title.lower())].append(sid)
    for (doc, tl), sids in by_title.items():
        sid, title = sids[0], sections[(doc, sids[0])]
        items.append({"task": "closed_book", "id": f"CB-{doc}-{sid}", "doc": doc, "section_id": sid, "title": title,
                      "accept": sids,
                      "prompt": (f"In the Nigerian National Standing Orders 2024 for {DOC_NAME[doc]}, which section "
                                 f"number covers '{title.title()}'? Answer only in the form: Section <number>")})
for c in chunks:
    ex = excerpt(c["text"], rng)
    if "index" in TASKS:
        items.append({"task": "index", "id": f"IX-{c['chunk_id']}", "doc": c["doc"], "section_id": c["section_id"],
                      "title": sections[(c["doc"], c["section_id"])],
                      "prompt": (f"Below is the table of contents of the Nigerian National Standing Orders 2024 for "
                                 f"{DOC_NAME[c['doc']]}, followed by an excerpt from the document.\n\n"
                                 f"TABLE OF CONTENTS:\n{toc[c['doc']]}\n\nEXCERPT:\n\"{ex}\"\n\n"
                                 "Which section is this excerpt from? Answer only in the form: Section <number> <title>")})
    if "mcq" in TASKS:
        true_t = sections[(c["doc"], c["section_id"])]
        others = [(sid, t) for (d, sid), t in sections.items() if d == c["doc"] and t.lower() != true_t.lower()]
        opts = rng.sample(others, 3) + [(c["section_id"], true_t)]
        rng.shuffle(opts)
        letters = "ABCD"
        gold = letters[[o[0] for o in opts].index(c["section_id"])]
        items.append({"task": "mcq", "id": f"MC-{c['chunk_id']}", "doc": c["doc"], "section_id": c["section_id"],
                      "title": true_t, "gold_letter": gold,
                      "prompt": (f"Excerpt from the Nigerian National Standing Orders 2024 for {DOC_NAME[c['doc']]}:\n"
                                 f"\"{ex}\"\n\nWhich section is it from?\n" +
                                 "\n".join(f"{l}. Section {sid} {t}" for l, (sid, t) in zip(letters, opts)) +
                                 "\n\nAnswer with the letter only.")})
if LIMIT:
    by_task = collections.defaultdict(list)
    for it in items:
        by_task[it["task"]].append(it)
    items = [it for t in TASKS for it in by_task[t][:LIMIT]]
    print(f"*** SMOKE TEST: {LIMIT} items per task ***")
print(len(sections), "sections,", len(chunks), "chunks ->", dict(collections.Counter(it["task"] for it in items)))
print("\nexample index prompt (end):\n..." + next(it for it in items if it["task"] != "closed_book")["prompt"][-500:])

## 2. Load raw N-ATLaS in 4-bit (from Drive if already saved)
First run: download from Hugging Face → quantise to 4-bit on the GPU → **save to Drive**. Later runs:
load the 4-bit copy from Drive directly. N-ATLaS ships with its end-of-turn token as padding; it is
replaced by a dedicated padding token (as Unsloth does in the other notebooks).

In [ ]:
import gc, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

for _name in ("model",):                                  # safe to re-run
    globals().pop(_name, None)
sys.last_type = sys.last_value = sys.last_traceback = None
gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
_free_gb = torch.cuda.mem_get_info()[0] / 2**30
assert _free_gb > 7, f"Only {_free_gb:.1f} GB GPU free: Runtime -> Restart session, then Run all"

if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("loading 4-bit N-ATLaS from", MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map={"": 0})
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
else:
    print("first run: downloading", BASE_MODEL, "from Hugging Face and quantising to 4-bit ...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0},
                                                 torch_dtype=torch.float16, token=HF_TOKEN)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    os.makedirs(MODEL_DIR, exist_ok=True)
    model.save_pretrained(MODEL_DIR); tok.save_pretrained(MODEL_DIR)
    size = sum(os.path.getsize(os.path.join(MODEL_DIR, f)) for f in os.listdir(MODEL_DIR)) / 2**30
    print(f"saved 4-bit copy to {MODEL_DIR} ({size:.1f} GB); next runs load from there")
    import shutil
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)   # free local disk

model.eval()
model.generation_config.max_length = None
tok.padding_side = "left"
if tok.pad_token is None or tok.pad_token == tok.eos_token:
    tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() else tok.eos_token

def chat(user_msgs, max_new=32):
    texts = [tok.apply_chat_template([{"role": "user", "content": u}], tokenize=False, add_generation_prompt=True)
             for u in user_msgs]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

print(chat(["In one sentence: what is a danger sign of pneumonia in a child?"], max_new=40)[0])

## 3. Ask the questions (resumable: answers are saved to Drive as they come)

In [ ]:
import time
OUT = os.path.join(RESULTS_DIR, "base_reference_answers.jsonl")
done = {}
if os.path.exists(OUT):
    for l in open(OUT, encoding="utf-8"):
        r = json.loads(l); done[r["id"]] = r["answer"]
todo = sorted([it for it in items if it["id"] not in done], key=lambda it: len(it["prompt"]))
print(f"{len(items)} items | {len(done)} already answered | {len(todo)} to ask")
MAX_NEW = {"closed_book": 16, "index": 32, "mcq": 6}
t0, nb = time.time(), (len(todo) + BATCH - 1) // BATCH
with open(OUT, "a", encoding="utf-8") as f:
    for b in range(nb):
        batch = todo[b * BATCH:(b + 1) * BATCH]
        answers = chat([it["prompt"] for it in batch], max_new=max(MAX_NEW[it["task"]] for it in batch))
        for it, a in zip(batch, answers):
            done[it["id"]] = a
            f.write(json.dumps({"id": it["id"], "answer": a}, ensure_ascii=False) + "\n")
        f.flush()
        if (b + 1) % 5 == 0 or b + 1 == nb:
            el = time.time() - t0
            print(f"  batch {b + 1}/{nb}  {el / 60:.1f} min  ETA {el / (b + 1) * (nb - b - 1) / 60:.0f} min", flush=True)

## 4. Score
* `number_ok`: the first section number in the answer equals the true one (e.g. `2.27`).
* `title_ok`: the answer names a section whose title matches the true title (RapidFuzz ≥ 80), or
  gives a number whose section has the same title (children/adult/elderly duplicates count).
* `mcq_ok`: the first letter A–D equals the correct option.
95% CIs by bootstrap over items.

In [ ]:
import numpy as np
from rapidfuzz import fuzz

NUM = re.compile(r"\b(\d{1,2}(?:\.\d{1,2}){1,2})\b")
title_of = {(d, s): t.lower() for (d, s), t in sections.items()}

def score(it, ans):
    s = {"task": it["task"], "doc": it["doc"], "id": it["id"], "answer": ans,
         "gold": f"{it['section_id']} {it['title']}"}
    if it["task"] == "mcq":
        m = re.search(r"\b([ABCD])\b", ans.strip().upper()[:20])
        s["mcq_ok"] = float(bool(m) and m.group(1) == it["gold_letter"])
        return s
    m = NUM.search(ans)
    num = m.group(1) if m else None
    s["number_ok"] = float(num in it.get("accept", [it["section_id"]]))
    text = NUM.sub("", ans).strip(" :-.").lower()
    same_title = num is not None and title_of.get((it["doc"], num)) == it["title"].lower()
    s["title_ok"] = float(same_title or (len(text) > 3 and fuzz.token_set_ratio(text, it["title"].lower()) >= 80))
    return s

assert score({"task": "index", "doc": "CHEW", "id": "x", "section_id": "2.27", "title": "WORM INFESTATION"},
             "Section 2.27 Worm Infestation")["number_ok"] == 1
assert score({"task": "mcq", "doc": "CHEW", "id": "y", "section_id": "2.3", "title": "FEVER", "gold_letter": "C"},
             "C. Section 2.3 Fever")["mcq_ok"] == 1

scored = [score(it, done[it["id"]]) for it in items if it["id"] in done]
boot = np.random.default_rng(0)
def ci(v):
    v = np.array(v, dtype=float)
    b = boot.choice(v, (2000, len(v))).mean(1)
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5), len(v)

rows = []
for task in TASKS:
    ss = [s for s in scored if s["task"] == task]
    if not ss:
        continue
    for metric in (["mcq_ok"] if task == "mcq" else ["number_ok", "title_ok"]):
        m, lo, hi, n = ci([s[metric] for s in ss])
        rows.append({"task": task, "metric": metric, "accuracy": round(m, 3), "ci_low": round(lo, 3),
                     "ci_high": round(hi, 3), "n": n, "chance": 0.25 if task == "mcq" else round(1 / len(sections), 3)})
print(f"{'task':12} {'metric':10} {'accuracy':>8}  95% CI          n   chance")
for r in rows:
    print(f"{r['task']:12} {r['metric']:10} {r['accuracy']:>8.3f}  [{r['ci_low']:.3f}, {r['ci_high']:.3f}]  {r['n']:4}  {r['chance']}")

import csv
SUMMARY = os.path.join(RESULTS_DIR, "base_reference_summary.csv")
DETAIL = os.path.join(RESULTS_DIR, "base_reference_scored.jsonl")
with open(SUMMARY, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
with open(DETAIL, "w", encoding="utf-8") as f:
    for s in scored:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print("\nsaved:", SUMMARY, "|", DETAIL)
for s in [s for s in scored if s["task"] == "index"][:3]:
    print(f"  gold: {s['gold']:45} | model: {s['answer'][:70]}")

## 5. What the baseline means for fine-tuning
Rules of thumb (the printout below applies them to your numbers):

* **closed_book `number_ok` near 0** (expected): N-ATLaS has never seen the 2024 Standing Orders, so a
  `SOURCE:` line can only be correct if the model **learns** the references (fine-tuning with the
  SOURCE line in every answer) or is **given** the text at answer time (retrieval/RAG).
* **index `number_ok` low but `mcq_ok` clearly above 25%**: the model understands the content but cannot
  map it to section numbers on its own: fine-tuning on Q&A whose answers end with the exact SOURCE
  helps; adding retrieval of the right chunk (BM25) at answer time helps most.
* **mcq near 25%**: the model barely relates excerpts to topics: fine-tuning on Standing Orders content
  is essential, and SOURCE accuracy should be judged on the fine-tuned model with retrieval.

In [ ]:
acc = {(r["task"], r["metric"]): r["accuracy"] for r in rows}
cb, ix, mc = acc.get(("closed_book", "number_ok")), acc.get(("index", "number_ok")), acc.get(("mcq", "mcq_ok"))
print("Baseline (raw N-ATLaS, no fine-tuning):")
if cb is not None:
    print(f"* knows section numbers by heart: {cb:.0%} ->",
          "as expected, it does not know the 2024 Standing Orders" if cb < 0.1 else "some prior knowledge")
if ix is not None:
    print(f"* finds the section from an excerpt + table of contents: {ix:.0%} ->",
          "poor: references must be taught (SOURCE in every training answer) and/or retrieved at answer time"
          if ix < 0.5 else "usable: retrieval + this model may already cite well")
if mc is not None:
    print(f"* picks the right section among 4: {mc:.0%} (chance 25%) ->",
          "understands content well" if mc >= 0.6 else "weak content-to-topic mapping: fine-tuning on the content is essential")

if UPLOAD_RESULTS:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for p in (SUMMARY, DETAIL):
        api.upload_file(path_or_fileobj=p, path_in_repo=f"results/{os.path.basename(p)}", repo_id=DATA_REPO,
                        repo_type="dataset", commit_message="01b: raw N-ATLaS reference baseline")
    print(f"uploaded to {DATA_REPO}/results/")